# Métricas de avaliação — classificação

**Objetivo:** num conjunto **desbalanceado**, mostrar como a acurácia engana, calcular a matriz de confusão e o `classification_report`, e desenhar a curva ROC.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Dados desbalanceados (5% de positivos)

In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, weights=[0.95, 0.05],
                           n_informative=5, random_state=SEMENTE)
print("proporcao de positivos:", round(y.mean(), 3))

## O classificador trivial 'tudo negativo'

In [ ]:
print("acuracia prevendo sempre 0:", round((y == 0).mean(), 3), " <- alta e inutil")

## Um modelo de verdade

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEMENTE)
clf = LogisticRegression(max_iter=500).fit(Xtr, ytr)
pred = clf.predict(Xte)
print(confusion_matrix(yte, pred))
print(classification_report(yte, pred, digits=3))

## Curva ROC e AUC

In [ ]:
from sklearn.metrics import roc_curve

proba = clf.predict_proba(Xte)[:, 1]
fpr, tpr, _ = roc_curve(yte, proba)
print("AUC:", round(roc_auc_score(yte, proba), 3))

figura = go.Figure()
figura.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", line=dict(color=AZUL), name="LogReg"))
figura.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                            line=dict(color=SUAVE, dash="dash"), name="aleatorio"))
figura.update_layout(title="Curva ROC", xaxis_title="taxa de falsos positivos",
                     yaxis_title="taxa de verdadeiros positivos", height=380,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercícios

**1.** Baixe o limiar para 0,2. O que acontece com recall e precisão da classe positiva?

**2.** Por que a AUC não muda ao alterar o limiar, mas a acurácia muda?

In [ ]:
# @title Solução
from sklearn.metrics import precision_score, recall_score
for thr in [0.5, 0.2]:
    p = (proba >= thr).astype(int)
    print("limiar", thr, ": recall", round(recall_score(yte, p), 3),
          "| precisao", round(precision_score(yte, p, zero_division=0), 3))
# A AUC integra TODOS os limiares, entao nao depende de um limiar especifico;
# acuracia, precisao e recall sao medidas em UM limiar.